# 00 Configure Pipeline

Edit the configuration cell, run it, and then run the downstream notebooks. The cell writes `outputs/pipeline_config.yaml`, which notebooks 00_1 through 05 load for shared parameters.


In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import default_config_path, write_pipeline_config


In [11]:
INCLUDE_DECISION = ["strict"] #["strict", "yes"]
CONFIG = {
    # Filesystem locations. Keep these relative to the repository root unless you need an external path.
    "paths": {
        "assets_dir": "assets",  # Static lookup tables and curated inputs.
        "output_dir": "outputs",  # All generated outputs. This directory is git-ignored.
        "parquet_dir": "outputs/parquets",  # Canonical parquet outputs shared by notebooks.
        "figure_dir": "outputs/figures",  # Saved plot outputs.
        "pathogens": "assets/bacteria_synonym_lookup.csv",  # Bacterial synonym lookup used for PubMed queries and GLASS matching.
        "pathogen_priority": "assets/counts_extended2.csv",  # Ranked pathogen source list; controls which pathogens enter collection first.
        "antibiotic_coarsening": "assets/antibiotic_coarsening_lookup.csv",  # Maps antibiotics to class/coarsened analysis levels.
        "antibiotic_synonyms": "assets/antibiotic_synonym_lookup.csv",  # Antibiotic synonym lookup used for queries and GLASS matching.
        "driver_category_coarsening": "assets/amr_driver_category_coarsening.csv",  # Maps fine LLM AMR categories to coarse driver labels.
        "glass_xlsx": "assets/GLASS list.xlsx",  # WHO GLASS source workbook used for GLASS pair comparisons.
        "glass_cleaned": "assets/glass_amr_cleaned.csv",  # Cached cleaned GLASS pair table written/read by GLASS helpers.
        "yearly_candidates": "outputs/parquets/candidate_amr_pair_year_counts.parquet",  # Candidate pair-year PMID lists before LLM screening.
        "yearly_pair_pmids": "outputs/parquets/candidate_amr_pair_pmid_links.parquet",  # Candidate pair-PMID link table for inspection/debugging.
        "yearly_screened": "outputs/parquets/yearly_amr_screened_pairs.parquet",  # LLM-accepted pair-year counts used by SI notebooks.
        "unified_decisions": "outputs/parquets/yearly_amr_unified_decisions.parquet",  # Row-level LLM decisions for candidate PMID/pathogen/antibiotic/year rows.
        "unified_failures": "outputs/parquets/yearly_amr_unified_failures.parquet",  # Retry state for rows that repeatedly fail LLM classification.
        "pathogen_background": "outputs/parquets/pathogen_tiab_counts.parquet",  # Background pathogen publication counts for SI normalization.
        "antibiotic_background": "outputs/parquets/antibiotic_tiab_counts.parquet",  # Background antibiotic publication counts for SI normalization.
        "interval_pmids": "outputs/parquets/pmids_results.parquet",  # Raw interval PubMed query results retained for audit/resume.
        "pubmed_metadata_cache": "outputs/parquets/pubmed_pmid_metadata.parquet",  # PMID publication-year/venue cache for PubMed results.
        "abstract_cache": "outputs/parquets/pmid_abstracts.parquet",  # Cached PubMed title/abstract text reused by LLM screening and cost estimates.
    },
    # PubMed collection and candidate generation controls used by notebook 04.
    "collection": {
        "pubmed_email": "xfcosta@gmail.com",  # Contact email sent to NCBI Entrez.
        "total_start_year": 1970,  # Full canonical collection start year.
        "total_end_year": 2026,  # Full canonical collection end year.
        "slice_start_year": 2000,  # Current run starts here; useful for partitioned collection.
        "slice_end_year": 2026,  # Current run ends here.
        "interval_year_gap": 5,  # PubMed query interval width in years.
        "num_pathogens": None,  # None means all ranked pathogens; integer means top N by pathogenic_index.
        "num_antibiotics": None,  # None means all antibiotics; integer means first N sorted antibiotics.
        "min_pathogenic_index": 0.0,  # Drop pathogens with pathogenic_index at or below this threshold.
        "extra_amr_keywords": [  # Additional PubMed query clauses ANDed into candidate retrieval.
            "\"antimicrobial resistance\"[Title/Abstract] OR \"antibiotic resistance\"[Title/Abstract]"  # Broad AMR text filter for titles/abstracts.
        ],
        "collection_verbose": 1,  # 0=silent, 1=batch summaries, 2=detailed interval logging.
        "collection_progress_every": 300,  # Print collection progress after this many units.
        "save_every": 500,  # Number of interval rows buffered before an append-only checkpoint.
        "pair_full_fetch_threshold": 9500,  # Fetch once and partition locally below this PubMed count.
        "summary_batch_size": 1000,  # PubMed ESummary batch size.
        "retrieve_batch_size": 9000,  # PubMed PMID retrieval batch size.
        "max_workers": 4,  # Concurrent pathogen-antibiotic pair workers.
        "max_pending_futures": 8,  # Bound queued pair work to limit memory and API pressure.
        "partial_interval_max": 1,  # Query only missing intervals when at most this many remain for a pair.
        "background_scope": "observed_entities",  # observed_entities is fast; all_entities is canonical/full.
        "background_fetch_threshold": 9500,  # Full-fetch threshold for background counts.
        "background_metadata_batch_size": 1000,  # PubMed metadata batch size.
    },
    # LLM screening controls used by notebook 04.
    "llm": {
        "model": "gpt-5-nano",  # OpenAI model used for abstract classification.
        "run_filter": True,  # False skips LLM calls and requires an existing compatible decisions parquet.
        "rerun_unified_decisions": False,  # True ignores active decisions and reruns with the current prompt/schema.
        "reasoning_effort": "minimal",  # GPT-5 reasoning effort for classification: minimal, low, medium, or high.
        "max_calls": None,  # Optional cap for quick tests; None means no cap.
        "max_workers": 2,  # Parallel LLM worker count.
        "batch_size": 8,  # Candidate rows processed per checkpoint batch.
        "max_failure_retries": 5,  # Retry-exhaustion threshold persisted in the failures parquet.
        "retry_cached_failures": True,  # True reprocesses rows saved as transient LLM failures, while still resuming completed rows.
        "abstract_fetch_batch_size": 200,  # PubMed abstract batch size before LLM calls.
        "prefetch_abstracts": True,  # Fetch missing abstracts ahead of LLM processing.
        "category_backfill_decisions": ["strict", "yes", "uncertain"],  # Existing decision labels eligible for category/schema backfill. Null means all.
        "include_explanatory_text": False,  # True asks the LLM for rationale/evidence text; False keeps outputs compact and cheaper.
        "progress_every": 200,  # LLM progress logging cadence.
        "screened_include_decisions": INCLUDE_DECISION, #["strict", "yes"],  # Decision labels carried forward into yearly_amr_screened_pairs.
    },
    # Cost-estimation controls used by notebook 02.
    "cost_estimation": {
        "pricing_by_model": {
            "gpt-4o-mini": {"input_usd_per_1m_tokens": 0.15, "output_usd_per_1m_tokens": 0.60, "cached_input_usd_per_1m_tokens": 0.075},
            "gpt-5-nano": {"input_usd_per_1m_tokens": 0.05, "output_usd_per_1m_tokens": 0.40, "cached_input_usd_per_1m_tokens": 0.005},
        },  # Update model prices from https://platform.openai.com/docs/pricing when needed.
        "batch_api_multiplier": 1.0,  # Use 1.0 for synchronous calls; adjust for Batch API discounts if used.
        "sample_size": 1000,  # Rows sampled for token extrapolation.
        "random_seed": 42,  # Reproducible sampling seed.
        "assumed_output_tokens_per_call": 140,  # Fallback output size if no decisions parquet exists.
        "use_cached_input_price": False,  # Conservative default because abstracts are mostly unique.
        "exclude_already_asked": True,  # Exclude candidate keys already present in the active decisions parquet.
        "project_to_configured_pathogens": True,  # Scale the estimate when the candidate parquet covers fewer pathogens than collection.num_pathogens.
    },
    # Scientific-interest model and plot controls used by notebook 05.
    "si": {
        "antibiotic_analysis_level": "antibiotic",  # Antibiotic grouping used for model inputs and outputs: antibiotic, Class, Simplified categories, High level categories, or Origin.
        "fit_start_year": 2000,  # First historical year used when fitting SI/count trends.
        "fit_end_year": 2023,  # Last observed year used for forecast-validation fitting and ranking thresholds.
        "publication_filter_end_year": 2025,  # Last year included when applying the AMR publication-volume filter.
        "curve_fit_end_year": 2025,  # Last complete year used for displayed/ranked production curves.
        "num_future_years": 5,  # Forecast horizon beyond fit_end_year for future SI/count predictions.
        "min_avg_amr_publications_per_year": 1.0,  # Legacy/alternative mean accepted AMR papers/year threshold for SI model inputs.
        "min_tot_amr_publications": 50,  # Minimum accepted AMR publications in the publication-filter window required for a pair to enter SI modeling.
        "min_amr_pair_publication_count": 3,  # Yearly pair counts below this threshold are unavailable to SI modeling and interpolation.
        "min_current_predicted_count": 10,  # Minimum predicted count at fit_end_year required for interesting-pair ranking.
        "min_current_amr_count": 10,  # Minimum observed/materialized AMR pair count at fit_end_year required for interesting-pair ranking.
        "min_num_years_with_valid_counts": 15,  # Minimum yearly AMR pair observations required for interpolation.
        "min_future_predicted_count": 10,  # Minimum predicted count at the future horizon required for interesting-pair ranking.
        "min_si_change": 0.01,  # Minimum projected SI increase required for an interesting-pair candidate.
        "min_post_fit_observed_years": 1,  # Minimum holdout years after fit_end_year needed to calculate forecast error filters.
        "max_post_fit_si_abs_pct_error_mean": 100.0,  # Maximum allowed mean absolute percent error for post-fit SI forecasts; higher is more permissive.
        "max_si_loess_residual_variance_mean": None,  # Optional in-sample SI LOESS residual variance ceiling; None disables this noise filter.
        "fallback_top_n": 10,  # Number of top-ranked pairs retained if strict interesting-pair filters return too few rows.
        "category_include_decisions": INCLUDE_DECISION,  # LLM decision labels counted in category-driver summaries.
        "driver_category_level": "coarse",  # Category plot level: original/fine categories or coarse grouped concepts.
        "driver_lookback_years": 5,  # Years before fit_end_year used to summarize recent driver-category changes.
        "max_driver_plots": 25,  # Maximum number of category-driver plots saved from notebook 05.
        "driver_pie_grid_columns": 5,  # Number of pie-chart columns in the all-pair driver composition PDF.
        "max_interesting_plots": 25,  # Maximum number of SI forecast plots saved for interesting pairs.
        "figure_format": "pdf",  # File format/extension for saved figures.
        "display_plots": True,  # Whether notebooks display generated plots inline as well as saving them.
        "max_display_plots": 10,  # Maximum number of generated plots displayed inline per plotting section.
        "fill_missing_amr_counts_with_zero": False,  # Keep missing pair-year AMR counts unavailable during plotting.
        "loess_frac": 0.6,  # LOESS smoothing span; larger values produce smoother curves.
        "loess_degree": 1,  # LOESS polynomial degree; 1 is locally linear smoothing.
        "tail_damping_years": None,  # Optional damping window for forecast tails; None disables damping.
        "shape_screen_plot_count": 10,  # Number of curve-shape screen examples plotted for each shape category.
    },
    # Interactive GLASS scatter controls used by notebook 06.
    "glass_interactive": {
        "antibiotic_analysis_level": "antibiotic",  # Antibiotic grouping for interactive GLASS comparisons.
        "num_future_years": 3,  # Forecast horizon shown in interactive GLASS scatter plots.
        "min_last_available_count": 7,  # Minimum latest observed accepted-AMR count for a pair to be displayed.
        "min_total_publications": 100,  # Minimum accepted-AMR publications summed over all years for a pair to be displayed.
        "min_count_abs_pct_error_mean": 0,  # Lower bound for count forecast error filter in the interactive plot.
        "max_count_abs_pct_error_mean": 75,  # Upper bound for count forecast error filter; lower values require more accurate historical forecasts.
        "min_counts_background": 10,  # X-axis threshold for shading the sparse-publication region; does not filter points.
        "min_abs_delta_si": 0.005,  # Minimum absolute SI shift required for display.
        "min_si_delta": None,  # Optional directional SI-change threshold; None disables this filter.
        "display_name": True,  # True labels points with readable pathogen-antibiotic names.
        "max_text_labels": 300,  # Maximum number of text labels drawn on the interactive plot.
        "width": 1400,  # Interactive plot width in pixels.
        "height": 900,  # Interactive plot height in pixels.
    },
    # Dedicated GLASS pair plot controls used by notebook 07.
    "glass_plots": {
        "antibiotic_analysis_level": "antibiotic",  # Antibiotic grouping for dedicated GLASS pair plots.
        "num_future_years": 5,  # Forecast horizon shown in dedicated GLASS pair curves.
        "min_avg_amr_publications_per_year": 5.0,  # Legacy/alternative mean accepted-AMR publications/year threshold for GLASS plots.
        "min_tot_amr_publications": 30,  # Minimum accepted-AMR publications in the fit window required before plotting a GLASS pair.
        "category_include_decisions": INCLUDE_DECISION,  # LLM decision labels counted in GLASS driver plots.
        "driver_category_level": "coarse",  # Driver category granularity for GLASS plots: coarse or original.
        "driver_lookback_years": 5,  # Recent window length used to summarize GLASS driver-category mix.
        "figure_format": "pdf",  # File format/extension for saved GLASS figures.
        "display_plots": True,  # Whether to display GLASS plots inline while saving them.
        "max_display_plots": 10,  # Maximum number of GLASS plots shown inline per section.
        "max_glass_plots": None,  # Optional cap for quick GLASS plot runs; None plots all eligible GLASS pairs.
    },
    # Pair PMID inspection controls used by notebook 08.
    "pair_pmids": {
        "pathogen": "Escherichia coli",  # Pathogen selected for PMID/title inspection.
        "antibiotic": 'Vancomycin', #"Meticillin", #"Vancomycin",  # Antibiotic selected for PMID/title inspection.
        "max_pmids_per_year": 10,  # Maximum PMID/title rows displayed per year for the selected pair.
    },
}


In [12]:
CONFIG_PATH = default_config_path(PROJECT_ROOT)
write_pipeline_config(CONFIG, CONFIG_PATH)
print(f"Wrote pipeline configuration to {CONFIG_PATH}")


Wrote pipeline configuration to /Users/f.costa/Code/Antimicrobial-Resistance/outputs/pipeline_config.yaml
